# Multivariate EDA — New Zealand Enrollments and NZ Funding

This notebook examines the joint relationship between NZ bachelor's-degree enrolments and NZ government funding (TEC SAC rates) by broad field of study, 2016–2025.

**Data sources:**
- Enrolments: `data/clean/NZ_bachelors_enrollments_2016_2025.csv`
- TEC SAC rates: `data/raw/NZ Funding/NZ_TEC_SAC_rates_all_years.csv` (government subsidy per EFTS, Level 2, excl. GST)

**Category alignment:** NZ enrolment data uses `category_key` (integer 1–11); TEC SAC data uses `category_code` (letter: A, B, C, H, I, J, N, V, …). The mapping between them is:

| category_key | field_of_study | TEC category_code |
|:---:|:---|:---:|
| 1 | Natural & Physical Sciences | V |
| 2 | Information Technology | B / C |
| 3 | Engineering & Related Technologies | N |
| 4 | Architecture & Building | N (shared) |
| 5 | Agriculture, Environmental & Related | H |
| 6 | Health | B |
| 7 | Education | I |
| 8 | Management & Commerce | J / A |
| 9 | Society & Culture | A |
| 10 | Creative Arts | A |

Note: TEC categories are cost-based, not field-based — one category may cover multiple NZSCED fields.

**Research question:** Does higher or lower NZ government subsidy (SAC rate) predict higher or lower enrolment levels or growth rates within a field? This is the NZ analogue of the AUS elasticity question explored in the AUS Enrollments-Funding EDA.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
sns.set_theme(style='whitegrid')

# Resolve paths
candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
enr_path = sac_path = None

for root in candidate_roots:
    c_enr = root / 'data' / 'clean' / 'NZ_bachelors_enrollments_2016_2025.csv'
    c_sac = root / 'data' / 'raw' / 'NZ Funding' / 'NZ_TEC_SAC_rates_all_years.csv'
    if c_enr.exists() and c_sac.exists():
        enr_path, sac_path = c_enr, c_sac
        break

if enr_path is None:
    raise FileNotFoundError(f'Datasets not found from: {Path.cwd()}')

df_enr = pd.read_csv(enr_path)
df_sac = pd.read_csv(sac_path)

print(f'Enrolments: {df_enr.shape}  | years: {sorted(df_enr["year"].unique())}')
print(f'SAC rates:  {df_sac.shape}  | years: {sorted(df_sac["year"].unique())} | categories: {sorted(df_sac["category_code"].unique())}')
display(df_enr.head())
display(df_sac.head())

## 1. Harmonize and Merge: Map TEC Categories to Enrolment Fields

Map each TEC `category_code` to the corresponding `category_key` in the enrolments file, then merge on `(category_key, year)`.

In [ ]:
# TEC category → category_key mapping (approximate; one TEC category can span multiple fields)
# Use the dominant / most representative mapping
CAT_TO_KEY = {
    'V': 1,   # Science → Natural & Physical Sciences
    'C': 2,   # IT (computing) → Information Technology
    'B': 6,   # Health sciences → Health  (B also covers some IT)
    'N': 3,   # Priority/Engineering → Engineering & Related Tech
    'H': 5,   # Agriculture/Environment → Agriculture, Environmental & Related
    'I': 7,   # Teaching → Education
    'J': 8,   # Business/Commerce → Management & Commerce
    'A': 9,   # Arts/Humanities/Social Sciences → Society & Culture (also Creative Arts & M&C)
}

sac_mapped = df_sac[df_sac['category_code'].isin(CAT_TO_KEY)].copy()
sac_mapped['category_key'] = sac_mapped['category_code'].map(CAT_TO_KEY)

# Use mean SAC rate per (category_key, year) where multiple codes map to same key
sac_by_key = (sac_mapped.groupby(['category_key','year'])['sac_rate_level2_excl_gst']
              .mean().reset_index().rename(columns={'sac_rate_level2_excl_gst': 'sac_rate'}))

# Enrolments: exclude total row
enr = df_enr[df_enr['field_of_study'] != 'Total (all fields)'].copy()

# Merge
merged = enr.merge(sac_by_key, on=['category_key','year'], how='inner')

print(f'Merged shape: {merged.shape}')
print(f'Years covered: {sorted(merged["year"].unique())}')
print(f'Fields covered: {merged["field_of_study"].nunique()}')
print('\nMissing values:')
print(merged.isna().sum())
display(merged.head(12))

## 2. Inspect the Merged Dataset

Overview of the joint enrolment + funding panel.

In [ ]:
merged_clean = merged.dropna(subset=['total_bachelors','sac_rate']).copy()

print('Merged dataset info:')
merged_clean.info()

print('\nSummary statistics:')
display(merged_clean[['total_bachelors','domestic_bachelors','international_bachelors','sac_rate']].describe())

# Overall correlation
r_total  = merged_clean['total_bachelors'].corr(merged_clean['sac_rate'])
r_dom    = merged_clean['domestic_bachelors'].corr(merged_clean['sac_rate'])
r_intl   = merged_clean['international_bachelors'].corr(merged_clean['sac_rate'])
print(f'\nr(total_bachelors, sac_rate):         {r_total:.3f}')
print(f'r(domestic_bachelors, sac_rate):      {r_dom:.3f}')
print(f'r(international_bachelors, sac_rate): {r_intl:.3f}')

## 3. SAC Rate vs Enrolment Level

Scatter plots of SAC rate against enrolment for each field and overall.

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(data=merged_clean, x='sac_rate', y='total_bachelors',
                hue='field_of_study', style='year', s=80)
sns.regplot(data=merged_clean, x='sac_rate', y='total_bachelors',
            scatter=False, color='black', line_kws={'linestyle': '--'})
plt.title('NZ SAC Rate vs Total Bachelor Enrolments')
plt.xlabel('SAC Rate Level 2 (NZD, excl. GST)')
plt.ylabel('Total Bachelor Enrolments')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

# Log-log scatter (demand elasticity framing)
merged_clean['log_sac']  = np.log(merged_clean['sac_rate'])
merged_clean['log_enr']  = np.log1p(merged_clean['total_bachelors'])

plt.figure(figsize=(10, 7))
sns.scatterplot(data=merged_clean, x='log_sac', y='log_enr',
                hue='field_of_study', s=80)
sns.regplot(data=merged_clean, x='log_sac', y='log_enr',
            scatter=False, color='black', line_kws={'linestyle': '--'})
plt.title('Log SAC Rate vs Log Enrolments (Elasticity View)')
plt.xlabel('log(SAC Rate)')
plt.ylabel('log(1 + Total Enrolments)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

# Facets: one panel per field
g = sns.FacetGrid(merged_clean, col='field_of_study', col_wrap=3,
                  height=3.2, aspect=1.2, sharey=False, sharex=False)
g.map_dataframe(sns.scatterplot, x='sac_rate', y='total_bachelors', hue='year', palette='viridis')
g.map_dataframe(sns.regplot, x='sac_rate', y='total_bachelors', scatter=False,
                color='black', line_kws={'linestyle': '--', 'linewidth': 1})
g.set_titles('{col_name}')
g.set_axis_labels('SAC Rate (NZD)', 'Enrolments')
g.fig.suptitle('SAC Rate vs Enrolments by Field', y=1.02)
plt.show()

## 4. SAC Rate Changes vs Enrolment Changes

Year-over-year changes in SAC rate and enrolments — tests whether funding changes are associated with same-year enrolment responses.

In [ ]:
chg = merged_clean.sort_values(['field_of_study','year']).copy()
chg['delta_sac_%'] = chg.groupby('field_of_study')['sac_rate'].pct_change() * 100
chg['delta_enr_%'] = chg.groupby('field_of_study')['total_bachelors'].pct_change() * 100
chg = chg.dropna(subset=['delta_sac_%','delta_enr_%'])

plt.figure(figsize=(9, 6))
sns.scatterplot(data=chg, x='delta_sac_%', y='delta_enr_%', hue='field_of_study', s=80)
sns.regplot(data=chg, x='delta_sac_%', y='delta_enr_%',
            scatter=False, color='black', line_kws={'linestyle': '--'})
plt.axhline(0, color='grey', ls=':', lw=1)
plt.axvline(0, color='grey', ls=':', lw=1)
plt.title('YoY % Change: SAC Rate vs Enrolments')
plt.xlabel('SAC Rate Change (%)')
plt.ylabel('Enrolment Change (%)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

r_chg = chg['delta_sac_%'].corr(chg['delta_enr_%'])
print(f'r(delta_sac, delta_enr) = {r_chg:.3f}')
print('Interpretation: a positive r would suggest higher funding → higher enrolments (supply-side stimulus).')
print('A near-zero or negative r suggests funding changes do not drive enrolment within a year.')

## 5. Time Trend Comparison: Enrolments and SAC Rates Side by Side

In [ ]:
# Normalise both series to 2019 = 100 for visual comparison
BASE = 2019
base_enr = merged_clean[merged_clean['year'] == BASE].set_index('field_of_study')['total_bachelors']
base_sac = merged_clean[merged_clean['year'] == BASE].set_index('field_of_study')['sac_rate']

merged_clean['enr_idx'] = merged_clean.apply(
    lambda r: r['total_bachelors'] / base_enr.get(r['field_of_study'], np.nan) * 100, axis=1)
merged_clean['sac_idx'] = merged_clean.apply(
    lambda r: r['sac_rate'] / base_sac.get(r['field_of_study'], np.nan) * 100, axis=1)

fields_present = merged_clean['field_of_study'].unique()
n_cols = 2
n_rows = (len(fields_present) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for ax, field in zip(axes, fields_present):
    fdata = merged_clean[merged_clean['field_of_study'] == field].sort_values('year')
    ax.plot(fdata['year'], fdata['enr_idx'], marker='o', label='Enrolments')
    ax.plot(fdata['year'], fdata['sac_idx'], marker='s', ls='--', label='SAC Rate')
    ax.axhline(100, color='grey', ls=':', lw=0.8)
    ax.axvline(2021, color='red', ls='--', lw=1, alpha=0.7)
    ax.set_title(field, fontsize=9)
    ax.set_ylabel('Index (2019=100)')
    ax.legend(fontsize=7)

for ax in axes[len(fields_present):]:
    ax.set_visible(False)

fig.suptitle(f'NZ Enrolments vs SAC Rate — Indexed ({BASE}=100), Red=2021 JRG (AUS)', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap: Enrolments × Funding

Pearson correlation matrix across key numeric fields.

In [ ]:
corr_cols = ['total_bachelors','domestic_bachelors','international_bachelors',
             'sac_rate','log_sac','log_enr','year']
corr_matrix = merged_clean[corr_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('NZ Enrolments × Funding — Correlation Matrix')
plt.tight_layout()
plt.show()

# Category-level correlation: enrolments vs SAC rate (within-field)
within_corr = (merged_clean.groupby('field_of_study')
               .apply(lambda g: g['total_bachelors'].corr(g['sac_rate']) if len(g) >= 3 else np.nan)
               .reset_index(name='r_enr_vs_sac')
               .sort_values('r_enr_vs_sac', ascending=False))
print('Within-field correlation: enrolments vs SAC rate')
display(within_corr)

## 7. Additional Statistical Checks

Nonlinearity, heterogeneity across fields, and basic elasticity estimate.

In [ ]:
try:
    from scipy import stats
except ImportError:
    stats = None

# Nonlinearity: log-log vs log-level
x = merged_clean['log_sac'].to_numpy()
y = merged_clean['log_enr'].to_numpy()

lin_c  = np.polyfit(x, y, 1)
quad_c = np.polyfit(x, y, 2)
lin_p  = np.polyval(lin_c, np.sort(x))
quad_p = np.polyval(quad_c, np.sort(x))

def r2(y_true, y_pred):
    return 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - y_true.mean())**2)

print(f'Log-log: linear R²={r2(y[np.argsort(x)], lin_p):.4f}  |  quadratic R²={r2(y[np.argsort(x)], quad_p):.4f}')
print(f'Elasticity estimate (log-log slope): {lin_c[0]:.4f}')
print('  → A 10% increase in SAC rate is associated with a {:.1f}% change in enrolments.'.format(lin_c[0] * 10))

# Heterogeneity across fields
groups = [g['total_bachelors'].dropna().values for _, g in merged_clean.groupby('field_of_study') if len(g) > 1]
if stats and len(groups) >= 2:
    lev_stat, lev_p = stats.levene(*groups, center='median')
    f_stat,   f_p   = stats.f_oneway(*groups)
    print(f"\nLevene's test on enrolments: F={lev_stat:.3f}, p={lev_p:.4g}")
    print(f'ANOVA on enrolments: F={f_stat:.3f}, p={f_p:.4g}')

## Data Characteristics & First-Order Effects

**Variables:** Panel of NZ field-level enrolments (total, domestic, international) and TEC SAC rates (Level-2 government subsidy per EFTS). Coverage is 2016–2025 for both, though the TEC category → NZSCED field mapping is approximate (multiple fields can share the same TEC cost category).

**Key interpretive challenge:** TEC cost categories are designed around **teaching cost** (lab equipment, contact hours), not enrolment demand. Higher SAC rates reflect higher teaching cost rather than higher student demand. The relationship between SAC rates and enrolments is therefore NOT a standard demand elasticity — it is more likely to reflect an equilibrium where universities offer programmes proportional to institutional capacity, subsidised by the government.

**Comparison with AUS:** In Australia, the JRG changed both the Commonwealth Contribution (equivalent to SAC rate) AND the student tuition fee simultaneously and discontinuously. In NZ, SAC rates change gradually and student fees are capped — making NZ a useful contrast to identify whether the AUS enrolment shift reflects the price shock rather than a common global trend.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style='whitegrid')

# Self-contained reload
candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
clean_dir = next((r / 'data' / 'clean' for r in candidate_roots if (r / 'data' / 'clean').exists()), None)
raw_nz    = next((r / 'data' / 'raw' / 'NZ Funding' for r in candidate_roots
                  if (r / 'data' / 'raw' / 'NZ Funding').exists()), None)

enr  = pd.read_csv(clean_dir / 'NZ_bachelors_enrollments_2016_2025.csv')
sac  = pd.read_csv(raw_nz / 'NZ_TEC_SAC_rates_all_years.csv')

CAT_TO_KEY = {'V':1,'C':2,'B':6,'N':3,'H':5,'I':7,'J':8,'A':9}
sac_m = sac[sac['category_code'].isin(CAT_TO_KEY)].copy()
sac_m['category_key'] = sac_m['category_code'].map(CAT_TO_KEY)
sac_k = sac_m.groupby(['category_key','year'])['sac_rate_level2_excl_gst'].mean().reset_index().rename(
    columns={'sac_rate_level2_excl_gst': 'sac_rate'})

enr_f = enr[enr['field_of_study'] != 'Total (all fields)'].copy()
merged = enr_f.merge(sac_k, on=['category_key','year'], how='inner').dropna()

print('=== NZ Enrolments × Funding — Variable Summary ===')
print(f'Merged: {merged.shape}  |  years: {sorted(merged["year"].unique())}')

r_raw, p_raw = stats.pearsonr(merged['total_bachelors'], merged['sac_rate'])
merged['log_enr'] = np.log1p(merged['total_bachelors'])
merged['log_sac'] = np.log(merged['sac_rate'])
r_log, p_log = stats.pearsonr(merged['log_enr'], merged['log_sac'])
print(f'r(enr, sac) raw  = {r_raw:.3f}  (p={p_raw:.4f})')
print(f'r(log_enr, log_sac) = {r_log:.3f}  (p={p_log:.4f})')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('NZ Enrolments × Funding — Data Characteristics', fontsize=12, fontweight='bold')

sc = axes[0].scatter(merged['log_sac'], merged['log_enr'],
                     c=merged['year'], cmap='viridis', alpha=0.8, s=60)
plt.colorbar(sc, ax=axes[0], label='Year')
axes[0].set_xlabel('log(SAC Rate)')
axes[0].set_ylabel('log(1+Enrolments)')
axes[0].set_title(f'A. Log-log scatter (r={r_log:.2f})')

axes[1].hist(merged['sac_rate'], bins=20, color='steelblue', edgecolor='white', alpha=0.7, label='SAC rate')
ax2 = axes[1].twinx()
ax2.hist(merged['total_bachelors'], bins=20, color='darkorange', edgecolor='white', alpha=0.5, label='Enrolments')
axes[1].set_xlabel('NZD / Enrolments')
axes[1].set_title('B. Overlapping distributions (scaled differently)')
axes[1].legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

### What Is Learned

1. **Category alignment:** TODO — note any fields that could not be matched (category_key without a SAC code mapping).
2. **Enrolment–funding relationship:** TODO — direction and strength of correlation. Is higher subsidy associated with higher or lower enrolment?
3. **Elasticity estimate:** TODO — report the log-log slope and its interpretation.
4. **YoY responsiveness:** TODO — do enrolment changes track SAC rate changes within a year?
5. **Implications for DiD:** TODO — if NZ enrolments are not responding to NZ funding changes, any NZ enrolment movement post-2021 is more cleanly attributable to global trends rather than domestic policy, strengthening NZ's control-group validity.